# Gemma 4 12B IT — 15 GB Colab QLoRA live test

This notebook performs a **real supervised fine-tuning test** of:

- **Model:** `unsloth/gemma-4-12b-it`
- **Library:** Unsloth
- **Method:** 4-bit QLoRA
- **Dataset:** `mlabonne/FineTome-100k`
- **Target hardware:** one NVIDIA T4-class Colab GPU with about 15 GB VRAM

The defaults are intentionally conservative so the first run has the best chance of fitting:

| Setting | Default |
|---|---:|
| Context length | 512 tokens |
| Training rows | 128 |
| LoRA rank | 4 |
| LoRA targets | language attention modules |
| Micro-batch | 1 |
| Gradient accumulation | 8 |
| Training steps | 10 |
| Base model precision | 4-bit |

This is **not full-parameter fine-tuning**. It trains LoRA adapters over a frozen 4-bit model.

### Before running

1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Run all cells from the top.
3. The first model download is large and uses substantial disk/RAM.
4. If Hugging Face requests authentication, add a Colab secret named `HF_TOKEN`.

At the end, the notebook prints an unambiguous **PASS/FAIL** result and the measured peak VRAM.

## 1. Install the current Unsloth stack

This cell follows the package structure used by the official Unsloth Gemma 4 notebook.  
After it finishes, continue normally; do not manually import an older `transformers` or `trl` version.

In [1]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -U unsloth
else:
    import torch
    version_match = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__))
    torch_minor = version_match.group(0) if version_match else "2.10"
    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(torch_minor, "0.0.34")

    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q --no-deps --upgrade "torchao>=0.16.0"

!pip install -q --no-deps "transformers==5.10.1" "tokenizers>=0.22.0,<=0.23.0"
!pip install -q "huggingface_hub>=1.5.0,<2.0"


## 2. Verify the GPU and define the memory-safe test profile

For a normal Colab T4, Python commonly reports roughly **14.7 GB** usable VRAM even though it is marketed as a 16 GB GPU.

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU found. In Colab select Runtime → Change runtime type → T4 GPU."
    )

gpu = torch.cuda.get_device_properties(0)
TOTAL_VRAM_GB = gpu.total_memory / 1024**3

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip())

print(f"\nCUDA device: {gpu.name}")
print(f"Usable VRAM reported by PyTorch: {TOTAL_VRAM_GB:.2f} GB")
print(f"CUDA devices visible: {torch.cuda.device_count()}")

if TOTAL_VRAM_GB < 13.5:
    raise RuntimeError(
        f"This notebook expects at least ~13.5 GB usable VRAM; only "
        f"{TOTAL_VRAM_GB:.2f} GB was detected."
    )

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


Tesla T4, 15360 MiB, 14910 MiB

CUDA device: Tesla T4
Usable VRAM reported by PyTorch: 14.56 GB
CUDA devices visible: 1


In [3]:
# Memory-safe live-test configuration.
# Change these only AFTER the default profile successfully passes.

MODEL_NAME = "unsloth/gemma-4-12b-it"
MAX_SEQ_LENGTH = 512
DATASET_ROWS = 128
MAX_STEPS = 10

LORA_R = 4
LORA_ALPHA = 4
FINETUNE_MLP = False       # False is safer on a 15 GB T4.
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

OUTPUT_DIR = "gemma4_12b_15gb_test_lora"
SEED = 3407

print({
    "model": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dataset_rows": DATASET_ROWS,
    "max_steps": MAX_STEPS,
    "lora_rank": LORA_R,
    "finetune_mlp": FINETUNE_MLP,
    "effective_batch_size": MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS,
})


{'model': 'unsloth/gemma-4-12b-it', 'max_seq_length': 512, 'dataset_rows': 128, 'max_steps': 10, 'lora_rank': 4, 'finetune_mlp': False, 'effective_batch_size': 8}


## 3. Optional Hugging Face token

The notebook first checks for a Colab secret named `HF_TOKEN`.  
A token is only needed if the model repository or license requires authentication.

In [4]:
HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

print("HF token found." if HF_TOKEN else "No HF_TOKEN secret found; trying public access.")


HF token found.


## 4. Load Gemma 4 12B IT in 4-bit, text-only mode

The important memory controls are:

- `load_in_4bit=True`
- `full_finetuning=False`
- `text_only=True`
- `offload_embedding=True` (best effort; Unsloth may disable it when embeddings are tied)
- context limited to 512 tokens

`text_only=True` avoids retaining unused multimodal towers for this text fine-tuning test.

In [5]:
from unsloth import FastModel

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
    text_only=True,
    offload_embedding=True,
    token=HF_TOKEN,
)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

load_allocated_gb = torch.cuda.memory_allocated() / 1024**3
load_reserved_gb = torch.cuda.memory_reserved() / 1024**3
load_peak_gb = torch.cuda.max_memory_reserved() / 1024**3

print("\nModel loaded successfully.")
print(f"Current allocated VRAM: {load_allocated_gb:.2f} GB")
print(f"Current reserved VRAM:  {load_reserved_gb:.2f} GB")
print(f"Peak reserved so far:   {load_peak_gb:.2f} GB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.8.9: Fast Gemma4_Unified patching. Transformers: 5.10.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4_unified won't work! Using float32.


Loading weights:   0%|          | 0/666 [00:00<?, ?it/s]

Gemma4UnifiedForCausalLM LOAD REPORT from: unsloth/gemma-4-12b-it
Key                                            | Status     |  | 
-----------------------------------------------+------------+--+-
model.vision_embedder.patch_ln2.bias           | UNEXPECTED |  | 
model.vision_embedder.patch_ln2.weight         | UNEXPECTED |  | 
model.vision_embedder.pos_embedding            | UNEXPECTED |  | 
model.vision_embedder.patch_dense.bias         | UNEXPECTED |  | 
model.vision_embedder.pos_norm.bias            | UNEXPECTED |  | 
model.embed_vision.embedding_projection.weight | UNEXPECTED |  | 
model.embed_audio.embedding_projection.weight  | UNEXPECTED |  | 
model.vision_embedder.patch_dense.weight       | UNEXPECTED |  | 
model.vision_embedder.pos_norm.weight          | UNEXPECTED |  | 
model.vision_embedder.patch_ln1.bias           | UNEXPECTED |  | 
model.vision_embedder.patch_ln1.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

Unsloth: Not offloading embeddings; this model ties embed_tokens to lm_head, so offloading saves no VRAM.

Model loaded successfully.
Current allocated VRAM: 7.13 GB
Current reserved VRAM:  7.21 GB
Peak reserved so far:   9.06 GB


## 5. Add trainable LoRA adapters

This default trains only the language model's attention modules. That is still real LoRA fine-tuning, while keeping the first 15 GB test conservative.

After the notebook passes, set `FINETUNE_MLP=True` to include the MLP projections too.

In [6]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_audio_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=FINETUNE_MLP,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()


trainable params: 5,332,992 || all params: 11,912,683,264 || trainable%: 0.0448


## 6. Load and format a known instruction dataset

`FineTome-100k` is also used in Unsloth's Gemma 4 supervised fine-tuning example.  
Only the first 128 rows are used here because the objective is to prove that training can run inside the VRAM limit.

In [7]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

dataset = load_dataset(
    "mlabonne/FineTome-100k",
    split=f"train[:{DATASET_ROWS}]",
)

dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    conversations = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        for conversation in conversations
    ]
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Applying Gemma 4 chat template",
)

print(dataset)
print("\nFormatted sample:\n")
print(dataset[0]["text"][:1500])


README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  117MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/128 [00:00<?, ? examples/s]

Applying Gemma 4 chat template:   0%|          | 0/128 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations', 'source', 'score', 'text'],
    num_rows: 128
})

Formatted sample:

<|turn>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. 

Furthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.

Finally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented d

## 7. Build the trainer

The effective batch size is `1 × 8 = 8`, but only one sequence is resident per micro-step.

In [9]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",

        max_length=MAX_SEQ_LENGTH,
        packing=False,
        padding_free=False,

        per_device_train_batch_size=MICRO_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_steps=1,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
    ),
)

from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

print("Trainer is ready.")
print(f"Training examples: {len(trainer.train_dataset)}")

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/128 [00:00<?, ? examples/s]

Unsloth: Auto-detected instruction_part = '<|turn>user\n' and response_part = '<|turn>model\n'


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128 [00:00<?, ? examples/s]

Unsloth: Removed 1 out of 128 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.
Trainer is ready.
Training examples: 127


## 8. Run the real fine-tuning test

This cell catches CUDA out-of-memory errors so the notebook can still print a useful FAIL report instead of ending with no diagnosis.

In [14]:
import gc
import torch
import torch.nn.functional as F

# Fix Gemma 4 / T4 mixed-dtype SDPA:
# query + key are float32, while shared value can remain bfloat16.
if not hasattr(F, "_gemma4_original_sdpa"):
    F._gemma4_original_sdpa = F.scaled_dot_product_attention

def _gemma4_safe_sdpa(query, key, value, *args, **kwargs):
    target_dtype = query.dtype

    if key.dtype != target_dtype:
        key = key.to(target_dtype)

    if value.dtype != target_dtype:
        value = value.to(target_dtype)

    return F._gemma4_original_sdpa(
        query,
        key,
        value,
        *args,
        **kwargs,
    )

F.scaled_dot_product_attention = _gemma4_safe_sdpa

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

START_ALLOCATED_GB = torch.cuda.memory_allocated() / 1024**3
START_RESERVED_GB = torch.cuda.memory_reserved() / 1024**3

TRAINING_OK = False
TRAINING_ERROR = None
trainer_stats = None

print(f"Allocated immediately before training: {START_ALLOCATED_GB:.2f} GB")
print(f"Reserved immediately before training:  {START_RESERVED_GB:.2f} GB")
print("\nStarting training...\n")

try:
    trainer_stats = trainer.train()
    TRAINING_OK = True

except Exception as exc:
    TRAINING_ERROR = f"{type(exc).__name__}: {exc}"
    raise

finally:
    PEAK_ALLOCATED_GB = torch.cuda.max_memory_allocated() / 1024**3
    PEAK_RESERVED_GB = torch.cuda.max_memory_reserved() / 1024**3

Allocated immediately before training: 7.22 GB
Reserved immediately before training:  7.38 GB

Starting training...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 127 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 5,332,992 of 11,912,683,264 (0.04% trained)


Step,Training Loss
1,2.186168
2,2.556644
3,2.189464
4,1.895746
5,1.660330
6,2.001830
7,1.550606
8,1.586741
9,1.778964
10,1.162655


## 9. Final 15 GB VRAM report

In [15]:
headroom_gb = TOTAL_VRAM_GB - PEAK_RESERVED_GB
utilization_pct = 100 * PEAK_RESERVED_GB / TOTAL_VRAM_GB

print("=" * 72)
print("GEMMA 4 12B IT — 15 GB COLAB QLORA TEST")
print("=" * 72)
print(f"GPU:                    {gpu.name}")
print(f"Usable GPU capacity:    {TOTAL_VRAM_GB:.2f} GB")
print(f"Peak allocated VRAM:    {PEAK_ALLOCATED_GB:.2f} GB")
print(f"Peak reserved VRAM:     {PEAK_RESERVED_GB:.2f} GB")
print(f"Reserved utilization:   {utilization_pct:.1f}%")
print(f"Remaining headroom:     {headroom_gb:.2f} GB")
print(f"Completed train steps:  {MAX_STEPS if TRAINING_OK else 0}/{MAX_STEPS}")

if trainer_stats is not None:
    print(f"Training runtime:       {trainer_stats.metrics.get('train_runtime', float('nan')):.2f} sec")
    print(f"Final reported loss:    {trainer_stats.metrics.get('train_loss', float('nan')):.6f}")

print("-" * 72)
if TRAINING_OK:
    print("PASS: Gemma 4 12B IT completed real 4-bit QLoRA training on this GPU.")
else:
    print("FAIL: The configured training test did not complete.")
print("=" * 72)


GEMMA 4 12B IT — 15 GB COLAB QLORA TEST
GPU:                    Tesla T4
Usable GPU capacity:    14.56 GB
Peak allocated VRAM:    12.61 GB
Peak reserved VRAM:     12.75 GB
Reserved utilization:   87.5%
Remaining headroom:     1.82 GB
Completed train steps:  10/10
Training runtime:       307.84 sec
Final reported loss:    1.856915
------------------------------------------------------------------------
PASS: Gemma 4 12B IT completed real 4-bit QLoRA training on this GPU.


## 10. Save the LoRA adapter

This saves only the small trainable adapter and tokenizer, not a merged 12B checkpoint.

In [16]:
import shutil
from pathlib import Path

if TRAINING_OK:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    archive_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    print(f"Saved adapter folder: {Path(OUTPUT_DIR).resolve()}")
    print(f"Saved ZIP archive:    {Path(archive_path).resolve()}")
else:
    print("Skipping save because training did not complete.")


Unsloth: Restored added_tokens_decoder metadata in gemma4_12b_15gb_test_lora/tokenizer_config.json.


Saved adapter folder: /content/gemma4_12b_15gb_test_lora
Saved ZIP archive:    /content/gemma4_12b_15gb_test_lora.zip


## 11. Test inference with the fine-tuned adapter

In [22]:
import gc
import torch
from transformers import TextStreamer

if TRAINING_OK:
    gc.collect()
    torch.cuda.empty_cache()

    ARCH = ["Gemma4UnifiedForConditionalGeneration"]

    objs = [
        model,
        getattr(model, "base_model", None),
        getattr(getattr(model, "base_model", None), "model", None),
        getattr(model, "model", None),
    ]

    for obj in objs:
        if obj is not None and hasattr(obj, "config"):
            obj.config.architectures = ARCH.copy()

    if hasattr(model, "for_inference"):
        model.for_inference()

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True

    model.eval()

    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": "Explain in three concise points why gradient accumulation is useful.",
        }],
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    print("Fine-tuned model response:\n")

    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=160,
            use_cache=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )

else:
    print("Skipping inference because training did not complete.")

Both `max_new_tokens` (=160) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuned model response:

<|channel>thought
<channel|>Here are three concise reasons why gradient accumulation is useful:

1.  **Memory Efficiency:** It allows you to train large models (or large batches) on hardware with limited GPU memory by accumulating gradients over multiple small "mini-batches" before updating the weights.
2.  **Simulates Large Batch Sizes:** It enables the use of larger effective batch sizes, which can lead to more stable gradient estimates and smoother convergence, without requiring the physical memory to process those samples simultaneously.
3.  **Cost-Effectiveness:** It provides a way to achieve the benefits of large-batch training (which often improves optimization) while staying within the hardware constraints of standard, consumer-grade, or existing enterprise hardware.<turn|>


## After the default test passes

Increase only one dimension at a time and restart the runtime between tests:

1. Set `FINETUNE_MLP=True`.
2. Increase `MAX_SEQ_LENGTH` from 512 to 768, then 1024.
3. Increase `LORA_R` from 4 to 8.
4. Increase `DATASET_ROWS` and `MAX_STEPS` for a meaningful training run.

Dataset size mostly changes **time**, not peak VRAM. Context length, micro-batch size, LoRA coverage, and model precision are the main memory controls.

For a serious run on the same 15 GB GPU, a reasonable next attempt is:

```python
MAX_SEQ_LENGTH = 768
DATASET_ROWS = 1000
MAX_STEPS = 100
LORA_R = 8
LORA_ALPHA = 8
FINETUNE_MLP = True
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
```

That larger profile is not guaranteed to fit every T4 runtime; the default profile is the actual compatibility test.